# Galakser Utforsker
Dette er en kladd for å teste klynging av ord-koordinasjoner direkte i en Notebook.

In [1]:
import network_explorer as ne
import json
import os
# Koble til databasen
db_path = 'koordinasjoner.db'
exp = ne.NetworkExplorer(db_path)

## Hent Nettverk og Linjegraf-klynger
Her kan vi teste søk. Funksjonen `get_neighborhood` tar inn `max_depth` (nivå) som bestemmer hvor mange ledd ut den skal lete. For eksempel `max_depth=2` eller `max_depth=3`.

In [2]:
%%time
# Parametere for søket
def neighbours_sqlite(ord_x = 'is',
tabell = 'avis_og', # prøv også 'avis_eller', 'bok_nob_og'
nivaa = 2,          # Dybde (1 = bare direkte naboer, 2 = naboers naboer)
topp_k = 20,        # Maks naboer per node
min_ratio = 100.0):   # Minimum Radon-Nikodym grense

# Kjør BFS søket i SQLite
    nodes, edges = exp.get_neighborhood(ord_x, tabell, max_depth=nivaa, top_k=topp_k, min_ratio=min_ratio)

    # Kjør linjegraf-klynging på resultatet (dette pakker nodene inn i dictionaries med overlappende 'groups')
    graf_data = exp.get_clustered_json_dict(nodes, edges)
    print(f"Fant {len(graf_data['nodes'])} ord og {len(graf_data['links'])} forbindelser.")
    return graf_data



CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 8.34 μs


In [3]:
%%time
# Parametere for søket
def neighbours_roaring(ord_x = 'is',
tabell = 'avis_og', # prøv også 'avis_eller', 'bok_nob_og'
nivaa = 2,          # Dybde (1 = bare direkte naboer, 2 = naboers naboer)
topp_n = 50,        # Hvilken Bitmap vi skal bruke
sample_k = None,    # Evt ned-sampling
seed = None):       # Seed for reproduserbarhet

# Kjør BFS søket i SQLite
    nodes, edges = exp.get_neighborhood_roaring(ord_x, tabell, max_depth=nivaa, top_n=topp_n, sample_k=sample_k, seed=seed)

    # Kjør linjegraf-klynging på resultatet (dette pakker nodene inn i dictionaries med overlappende 'groups')
    graf_data_bm = exp.get_clustered_json_dict(nodes, edges)
    print(f"Fant {len(graf_data_bm['nodes'])} ord og {len(graf_data_bm['links'])} forbindelser.")
    return graf_data_bm

CPU times: user 1e+03 ns, sys: 0 ns, total: 1e+03 ns
Wall time: 4.05 μs


In [4]:
#graf_data_bm['nodes']

## Visualisering
Dataene ligger nå perfekte for bruk med for eksempel `pyvis` for å tegne grafen interaktivt her inne i notebooken.

In [5]:
import matplotlib
import networkx as nx
import matplotlib.pyplot as plt

In [6]:
def draw(graf_data_bm):
    G = nx.Graph()
    
    # Legg til nodene
    for n in graf_data_bm['nodes']:
        # SIKRING: Hvis groups allerede er en liste/tuple, gjør vi den om til en sortert streng.
        # Hvis det er en vanlig streng/int fra før, fungerer det likevel.
        if isinstance(n['groups'], (list, tuple, set)):
            gruppe_id = " + ".join(sorted([str(g) for g in n['groups']]))
        else:
            gruppe_id = str(n['groups'])
            
        G.add_node(n['id'], label=n['label'], groups=gruppe_id)
    
    # Legg til kantene
    for e in graf_data_bm['links']:
        G.add_edge(e['source'], e['target'], weight=e['ratio'])
    
    # --- GENERER UNIKE FARGER FOR KOMBINASJONENE ---
    unike_kombinasjoner = sorted(list(set(nx.get_node_attributes(G, 'groups').values())))
    
    # Bruker oppdatert Matplotlib-syntaks (matplotlib.colormaps) for å unngå advarsler
    cmap = plt.colormaps['tab20'].resampled(len(unike_kombinasjoner))
    gruppe_til_farge = {gruppe: cmap(i) for i, gruppe in enumerate(unike_kombinasjoner)}
    
    # Lag fargelisten i riktig node-rekkefølge
    node_farger = [gruppe_til_farge[G.nodes[n]['groups']] for n in G.nodes]
    
    # Tegn grafen
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, k=0.6, iterations=50)
    
    nx.draw(
        G, 
        pos=pos, 
        with_labels=True, 
        node_color=node_farger, 
        font_size=10, 
        node_size=500
    )
    
    plt.show()


In [7]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

def show_clusters(graf_data_bm):
    G = nx.Graph()
    
    # Legg til nodene og håndter flere grupper
    for n in graf_data_bm['nodes']:
        if isinstance(n['groups'], (list, tuple, set)):
            gruppe_id = " + ".join(sorted([str(g) for g in n['groups']]))
        else:
            gruppe_id = str(n['groups'])
            
        G.add_node(n['id'], label=n['label'], groups=gruppe_id)
    
    # Legg til kantene
    for e in graf_data_bm['links']:
        G.add_edge(e['source'], e['target'], weight=e['ratio'])
    
    # --- 1. GRUPPER NODENE FOR UTSKRIFT (CLUSTERE) ---
    clustere = defaultdict(list)
    for n_id, data in G.nodes(data=True):
        # Bruker labelen hvis den finnes, ellers ID-en til noden
        navn = data.get('label', n_id)
        clustere[data['groups']].append(navn)
    
    # Skriv ut clustrene i terminalen
    print("=== OVERSIKT OVER CLUSTERE ===")
    for gruppe, noder in sorted(clustere.items()):
        print(f"\nKlynge: [{gruppe}] ({len(noder)} noder)")
        print(f"  -> {', '.join(noder)}")
    print("\n" + "="*30)
    



In [12]:
%%time
graf_data_bm = neighbours_roaring(ord_x="is",nivaa=3, topp_n=100, sample_k=10, seed=42)

CPU times: user 4.38 s, sys: 895 ms, total: 5.28 s
Wall time: 6.89 s


KeyboardInterrupt: 

In [9]:
draw(graf_data_bm)

ModuleNotFoundError: No module named 'scipy'

<Figure size 1200x1200 with 0 Axes>

In [10]:
show_clusters(graf_data_bm)

=== OVERSIKT OVER CLUSTERE ===

Klynge: [0] (53 noder)
  -> 2812, agn, akkar, avgaatt, avgikk, bjørkebladene, fabriksild, fabrikvare, fersk, ferskfisk, fersksild, flomlys, folgebrev, forhjuls-, frostfrie, fuktighet, følgebrev, følgebrevsbok, glo, hermetikksild, husbrug, isa, isdannelse, isfjerner, ising, kald, kaldt, kanner, kondensvann, lodde, madsild, matsild, melkespann, pappesker, revmatsild, rimdannelse, saltsild, skjærsild, smaasild, solar, solarfyringsolje, sommernattsmørke, stov, tetnet, tomflasker, tomtonder, tomtønder, tomtønner, treull, tønner, urenheter, vannskade, vike

Klynge: [0 + 1] (5 noder)
  -> agnsild, dugg, frosi, kjøving, tomkasser

Klynge: [0 + 1 + 13 + 9] (1 noder)
  -> underkjølt

Klynge: [0 + 1 + 16] (1 noder)
  -> duggdannelse

Klynge: [0 + 10 + 7] (1 noder)
  -> smeltevann

Klynge: [0 + 16] (1 noder)
  -> isdannelser

Klynge: [0 + 16 + 7 + 9] (1 noder)
  -> snømengder

Klynge: [0 + 6] (3 noder)
  -> agnskjell, dråper, skjell

Klynge: [0 + 6 + 7] (1 noder)
  